In [3]:
import pandas as pd 
import json 
import lazop
import os
from datetime import datetime, timedelta
appkey = int(os.environ["APPKEY"])
url_callback = os.environ["URLCALLBACK"]
appSecret = os.environ["APPSECRET"]
gateway_url = os.environ["GATEWAYURL"]

lazada_client = lazop.LazopClient(gateway_url, appkey ,appSecret)
pd.set_option('display.max_columns', None)



In [4]:
def auth():
    # url_callback = 'url_callback'
    # app_key = ''
    auth_url = f'https://auth.lazada.com/oauth/authorize?response_type=code&force_auth=true&redirect_uri={url_callback}&client_id={appkey}'
    print(auth_url)

auth()


https://auth.lazada.com/oauth/authorize?response_type=code&force_auth=true&redirect_uri=https://asia-southeast1-xongdur-dashboard-404505.cloudfunctions.net/function-1&client_id=128859


In [5]:
auth_code = ''

In [6]:
import json
import time
import lazop
from google.cloud import storage

def get_token():

    bucket_name = "token_shopee_lazada_file"
    file_name = "token_lazada_new.json"

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(file_name)

    request = lazop.LazopRequest('/auth/token/create')
    request.add_api_param('code', auth_code)

    response = lazada_client.execute(request)

    if response.code != "0":
        raise Exception(f"Lazada API error: {response.body}")

    response_dict = response.body

    access_token = response_dict['access_token']
    refresh_token = response_dict['refresh_token']
    expires_in = response_dict['expires_in']

    expires_at = int(time.time()) + expires_in

    data = {
        "access_token": access_token,
        "refresh_token": refresh_token,
        "expires_in": expires_in,
        "expires_at": expires_at
    }

    blob.upload_from_string(
        json.dumps(data),
        content_type="application/json"
    )

    print("Token saved to GCS")

get_token()

c:\Users\bell2\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Exception: Lazada API error: {'type': 'ISV', 'code': 'IncompleteSignature', 'message': 'The request signature does not conform to platform standards', 'request_id': '2151f12817732036275176835', '_trace_id_': '2151e14217732036275144506e0582'}